In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner='DataTalksClub',
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions=["md"],
    filename_filter= lambda path: '/lessons/' in path,
    
)

files = reader.read()

In [2]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)
    
len(documents)

72

In [3]:
from minsearch import Index

index = Index(
    text_fields=['content'],
    keyword_fields=['filename']
)
index.fit(documents)

In [4]:
index.search("How does the agentic loop keep calling the model until it stops?")

[{'content': '# The Agentic Loop\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we did function calling by hand. We sent a\nmessage and got back a function call. We ran it, sent the result back,\nand got the answer.\n\nThat works for one function call. It breaks down when the model wants\nto search several times, or when the first search misses the answer.\nWe don\'t know in advance how many calls the model will want. So we\nneed a loop that keeps calling the model and running tools until it\'s\ndone. An agent is exactly that.\n\n## Anatomy of an agent\n\nWith the LLM in the driver\'s seat, we have an agent. It\'s an AI\nassistant whose goal is to help the user.\n\nAn agent has three parts:\n\n- Instructions, the role and behavior we want. We pass this as the\n  `developer` message. The better the instructions, the better the\n  agent helps.\n- Tools, the functions the agent can call to carry

In [5]:
# !wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py

In [6]:
import os
from dotenv import load_dotenv

In [7]:
from openai import OpenAI
or_client = OpenAI(
    base_url = "https://openrouter.ai/api/v1",
    api_key = os.getenv('OPENROUTER_API_KEY')
)

In [8]:
from rag_helper import RAGBase
rag = RAGBase(index,or_client,'openrouter/free')
rag.rag("How does the agentic loop keep calling the model until it stops?")

'The agentic loop keeps calling the model until the model returns a turn that contains **no function calls**.  \n\nIn code the loop is:\n\n```python\nwhile True:\n    response = openai_client.responses.create(\n        model=..., input=messages, tools=[search_tool]\n    )\n    … # add the response to the conversation, run any function calls\n\n    has_function_calls = any(item.type == "function_call" for item in response.output)\n    if not has_function_calls:\n        break   # the model is done, return the final answer\n```\n\nThe `has_function_calls` flag is set true whenever the model’s response includes a `function_call`. After the loop finishes, the last message from the model (the one that had no function call) is the final answer. If you want safety nets you can add a max‑iteration count, a token budget, etc., but the core stopping condition is simply “no function calls this turn.”'

In [9]:
rag.tokens

ResponseUsage(input_tokens=7197, input_tokens_details=InputTokensDetails(cached_tokens=64), output_tokens=209, output_tokens_details=OutputTokensDetails(reasoning_tokens=3), total_tokens=7406, cost=0, is_byok=False, cost_details={'upstream_inference_cost': 0, 'upstream_inference_input_cost': 0, 'upstream_inference_output_cost': 0})

In [13]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)
len(chunks)

295

In [17]:
index.fit(chunks)

In [16]:
rag = RAGBase(index,or_client,'openrouter/free')
rag.rag("How does the agentic loop keep calling the model until it stops?")

'\nBased on the context provided, here\'s how the agentic loop keeps calling the model until it stops:\n\n## The Loop Mechanism\n\nThe agentic loop uses a `while True:` infinite loop with a specific **exit condition** based on whether the model makes function calls:\n\n1. **Flag System**: A boolean flag `has_function_calls` is initialized to `False` at the start of each iteration\n\n2. **Model Invocation**: The model is called via `openai_client.responses.create()` with the current conversation history and available tools\n\n3. **Response Analysis**: After receiving the response, the code examines each item in `response.output`:\n   - If an item is a `function_call`, the corresponding tool is executed and `has_function_calls` is set to `True`\n   - If an item is a `message`, it\'s displayed as the assistant\'s response\n\n4. **Exit Condition**: At the end of each iteration, the loop checks:\n   ```python\n   if has_function_calls == False:\n       break\n   ```\n   This means: **"If th

In [18]:
rag.tokens

ResponseUsage(input_tokens=2483, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=604, output_tokens_details=OutputTokensDetails(reasoning_tokens=254), total_tokens=3087, cost=0, is_byok=False, cost_details={'upstream_inference_cost': 0, 'upstream_inference_input_cost': 0, 'upstream_inference_output_cost': 0})

In [22]:
def search(query :str ) -> dict[str,str]:
    """
     Search the course Database for entries matching the given query
    """
    
    boost_dict = {'content':1}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

In [19]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [23]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [25]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the course Database for entries matching the given query',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [26]:
INSTRUCTIONS = '''
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
'''

In [36]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools = agent_tools,
    developer_prompt=INSTRUCTIONS,
    chat_interface= chat_interface,
    llm_client=OpenAIClient(client = or_client,model='openai/gpt-oss-120b')
)

In [38]:
runner.loop(
    prompt = "How does the agentic loop work, and how is it different from plain RAG?",
    callback=callback,
)

-> Response received


-> Response received


-> Response received


-> Response received


-> Response received


Aspect,Agentic Loop,Plain RAG
Control flow,"Dynamic, iterative: model may call tools many times until done.","Fixed, single pass: one retrieval → one generation."
Tool usage,"Model decides when and which tool to invoke (search, web‑lookup, database, etc.).",Retrieval is the only “tool” and is performed once before the LLM call.
Conversation history,"Full message history is kept and sent back on each turn, enabling self‑correction and multi‑turn reasoning.",No history beyond the single prompt built from the one retrieval.
Cost tracking,Tokens/cost are accumulated across all turns (result.cost).,"Cost is just the tokens of the single LLM call (plus the one retrieval call, if billed)."
Typical use case,"Complex tasks where the steps are not known upfront (e.g., research, planning, troubleshooting).",Simple question‑answering where you only need to fetch relevant documents once.


LoopResult(new_messages=[EasyInputMessage(content='\nYour task is to answer questions from the course participants\nbased on the provided context.\n\nUse the context to find relevant information and provide accurate\nanswers. If the answer is not found in the context,\nrespond with "I don\'t know."\n', role='developer', phase=None, type=None), EasyInputMessage(content='How does the agentic loop work, and how is it different from plain RAG?', role='user', phase=None, type=None), ResponseReasoningItem(id='rs_tmp_ndmyn7krln', summary=[], type='reasoning', content=[Content(text="We need to answer based on provided context, but we haven't seen context. We need to search database. Use function search.", type='reasoning_text')], encrypted_content=None, status='completed', format='unknown'), ResponseFunctionToolCall(arguments='{\n  "query": "agentic loop plain RAG difference"\n}', call_id='chatcmpl-tool-9ec2a9bdc05b36b0', name='search', type='function_call', id='fc_tmp_u2tgf6n058', namespace=N